# 第20周 · Capstone 汇报与课程总结

**本周安排**：S1–S3 是 Capstone 汇报（每人 15 分钟陈述 + 5 分钟问答），其余同学用同伴评审表评审；S4 前半是汇报缓冲与讨论，后半是课程总结——一张数学概念图、一张"下一步学什么"的路线图，以及每人一份个人反思。

20 周前，你可能还没写过一行代码。今天你要站在同学面前，讲清楚一个你自己提出、自己设计、自己用证据回答的问题——带基线、带对照、带消融、可复现。评审看的不是你的模型分数有多高，而是**你的证据能不能支持你的结论**；一个设计严谨的阴性结果，同样是好的研究。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from zero2researcher.checks import report

---
## S1–S3 · Capstone 汇报

### 汇报流程

- 每人 **15 分钟陈述 + 5 分钟问答**。第 12 分钟举牌提示，第 15 分钟停止陈述——在规定时间内讲完，本身就是汇报能力的一部分。
- 建议的幻灯片结构（约 10 张）：
  1. 题目与**一句话结论**；
  2. 问题与 bit flip：领域里的默认假设是什么，你把它翻转成了什么，可证伪的假设是什么；
  3. 相关工作（1 张就够）；
  4. 方法与实验设置：数据与划分、基线、科学 / 冗余 / 固定变量、种子数；
  5. 主结果（带误差棒，标题写结论）；
  6. 消融与失败案例；
  7. 局限与下一步；
  8. 复现方式与 AI 助手使用声明（可以放在备用页）。
- 问答：先由两位**深度评审人**各问一个问题（来自评审表第 4 节），再开放提问。回答不知道的问题时，说"我不知道，我会这样去查"，比硬答好。

### 同伴评审表怎么用

评审表在 `templates/peer_review_form.md`，字段统一参照 **NeurIPS 2025** 的审稿表（Summary、Strengths and Weaknesses、Questions、Limitations，Quality / Clarity / Significance / Originality 四项 1–4 分，总体评分 1–6、置信度 1–5；NeurIPS 2024 及更早用的是 Soundness / Presentation / Contribution 三项，见评审表开头的说明），外加本课程的两项：**新颖性——bit flip 是否成立**，以及**实验设计检查表**。

- 每位同学**深度评审两场**（下面随机分配）：汇报前读对方的报告初稿，听汇报时补充，汇报结束后 24 小时内交表；
- 其余每场汇报只填表末的"汇报快评"一行（约 2 分钟）；
- 原则：对事不对人；先复述、再评价；每条批评都指出具体位置（第几节、哪张图），并给出一个可操作的改进建议。同伴评审不计入被评审者的成绩，但评审的质量计入你自己的参与表现。

### 汇报顺序与评审分配

顺序随机生成（固定种子，可复现）；每场的两位深度评审人是顺序里紧随其后的两位同学——每人恰好深度评审两场，且不会评审自己。把 `students` 换成班级名单再运行：

In [ ]:
students = [f"学生{i:02d}" for i in range(1, 13)]      # 换成班级名单
rng = np.random.default_rng(2026)
order = [str(s) for s in rng.permutation(students)]
n = len(order)

t = dt.datetime(2026, 1, 1, 9, 0)        # 只用时间部分：9:00 开始
rows = []
for i, presenter in enumerate(order):
    if i > 0 and i % 4 == 0:
        t += dt.timedelta(minutes=10)    # 每 4 场休息 10 分钟
    rows.append({"#": i + 1, "start": t.strftime("%H:%M"), "presenter": presenter,
                 "deep reviewers": f"{order[(i + 1) % n]}, {order[(i + 2) % n]}"})
    t += dt.timedelta(minutes=20)        # 15 分钟陈述 + 5 分钟问答
schedule = pd.DataFrame(rows)

load = {s: sum(s in r.split(", ") for r in schedule["deep reviewers"]) for s in students}
report("评审分配：每人恰好深度评审两场，且不评审自己",
       all(v == 2 for v in load.values()) and
       all(p not in r for p, r in zip(schedule["presenter"], schedule["deep reviewers"])))
print(f"{n} 场汇报，预计 {schedule['start'].iloc[0]}–{t.strftime('%H:%M')}")
schedule

---
## S4 · 课程总结

### ① 数学概念图

整门课的数学可以串成一条线：**向量 → 梯度 → 链式法则 → 概率 → KL 散度 → 向量场**。每个概念都在某一周第一次出现，又在后面的模型里反复使用：

In [ ]:
main = [("Vectors &\nmatrices\n(W4)", "Linear models,\nembeddings\n(W4, W11)"),
        ("Gradients\n(W4, W6)", "Gradient descent,\nAdam (W4, W9)"),
        ("Chain rule\n(W6, W7)", "Backprop,\nmicrograd (W7, W8)"),
        ("Probability &\nlikelihood\n(W5, W14)", "Cross-entropy,\npolicy gradient\n(W5, W18)"),
        ("KL divergence\n(W14, W15)", "VAE: ELBO\n(W15)"),
        ("Vector fields\n(W16, W17)", "Flow Matching,\nDDPM (W16, W17)")]
links = ["stack partial\nderivatives", "gradients of\ncompositions", "train models that\noutput probabilities",
         "compare two\ndistributions", "move one distribution\nonto another"]

fig, ax = plt.subplots(figsize=(13, 4.8))
box_main = dict(boxstyle="round,pad=0.45", fc="#dbe9f6", ec="#2b6cb0", lw=1.5)
box_app = dict(boxstyle="round,pad=0.4", fc="#f7f7f7", ec="#888888", lw=1)
for x, (concept, used_in) in enumerate(main):
    ax.text(x * 2.2, 1.0, concept, ha="center", va="center", fontsize=10, fontweight="bold", bbox=box_main)
    ax.text(x * 2.2, -0.35, used_in, ha="center", va="center", fontsize=8.5, bbox=box_app)
    ax.annotate("", xy=(x * 2.2, -0.08), xytext=(x * 2.2, 0.62),
                arrowprops=dict(arrowstyle="->", color="#888888", ls="--"))
for x, label in enumerate(links):
    ax.annotate("", xy=(x * 2.2 + 1.45, 1.0), xytext=(x * 2.2 + 0.75, 1.0),
                arrowprops=dict(arrowstyle="-|>", color="#2b6cb0", lw=1.8))
    ax.text(x * 2.2 + 1.1, 1.38, label, ha="center", va="bottom", fontsize=8, color="#2b6cb0")
ax.add_patch(FancyArrowPatch((2.2, 1.45), (11.0, 1.45), connectionstyle="arc3,rad=-0.18",
                             arrowstyle="-|>", mutation_scale=14, color="#c05621", lw=1.2, ls=":"))
ax.text(6.6, 2.35, "a gradient field is a vector field: following its negative is gradient descent",
        ha="center", fontsize=8.5, color="#c05621")
ax.set_xlim(-1.0, 12.0)
ax.set_ylim(-0.9, 2.6)
ax.axis("off")
ax.set_title("Six ideas carry the whole course: from vectors to vector fields", fontsize=12)
plt.tight_layout()
plt.show()

| 概念 | 一句话 | 第一次出现 | 后来用在 |
| --- | --- | --- | --- |
| 向量与矩阵 | 数据、参数、嵌入都是向量；一层网络是一次矩阵乘法 | 第 4 周数学工作坊 1 | 线性回归、注意力的 $QK^\top$、GCN 的 $\hat{A}XW$ |
| 梯度 | 把每个参数的偏导数排成一个向量，指向函数上升最快的方向 | 第 4、6 周 | 梯度下降、Adam、梯度检查 |
| 链式法则 | 复合函数的导数 = 各层导数相乘；反向传播就是把它高效地做一遍 | 第 6 周数学工作坊 2、第 7 周 | micrograd、PyTorch autograd |
| 概率与似然 | 模型输出一个分布；训练 = 让观测数据的似然最大 | 第 5、14 周 | 交叉熵损失、语言模型、REINFORCE |
| KL 散度 | 衡量两个分布差多远：$\mathrm{KL}(q\Vert p)=\mathbb{E}_q[\log q - \log p]$ | 第 14 周数学工作坊 3 | VAE 的 ELBO、RLHF 中约束策略不要偏离太远 |
| 向量场 | 给空间中每一点指定一个速度；沿着它走，就把一个分布搬运成另一个 | 第 16 周 | Flow Matching、DDPM 的去噪方向 |

### ② 你现在能做什么，下一步学什么

**你现在能做的**：

- 读入、清洗、可视化一份表格或图像数据，建立从多数类到强基线的基线阶梯；
- 从零写 PyTorch 训练循环，用诊断配方排查"训不动"的模型；
- 搭建并训练 CNN、Transformer、GNN、VAE、Flow Matching / DDPM 和 REINFORCE；微调预训练模型（线性探针、全量微调、LoRA）；
- 写一个会用工具的 Agent，并为它建评测集、做失败分析；
- 设计受控实验（一个科学变量、调好的冗余变量、≥ 3 个种子、均值 ± 标准差、消融），读论文、复现一张图并量化差异，写 4 页报告、做 15 分钟汇报、给同伴写审稿意见。

**下一步**（链接均于 2026-10 核实可访问）：

| 方向 | 课程 / 资源 | 你已经具备的 | 建议从哪里开始 |
| --- | --- | --- | --- |
| 深度学习与计算机视觉 | [Stanford CS231n](https://cs231n.stanford.edu/) | 第 7–10 周：反向传播、训练循环、CNN | 前几次作业：从零实现网络的前向 / 反向传播与优化，和第 7–9 周一一对应 |
| 从零构建语言模型 | [Stanford CS336: Language Modeling from Scratch](https://cs336.stanford.edu/) | 第 11–12 周的 Transformer 与字符级 GPT；第 14 周的微调 | 第一次作业：BPE 分词器 + Transformer 语言模型的训练（Capstone 参考选题 5 的完整版） |
| 生成模型 | [MIT 6.S184：Flow Matching 与扩散模型](https://diffusion.csail.mit.edu/) | 第 15–17 周：VAE、Flow Matching、DDPM | 讲义里的常微分 / 随机微分方程视角，把第 16–17 周的向量场与概率路径讲完整 |
| 强化学习 | [Berkeley CS 185/285: Deep Reinforcement Learning](https://rail.eecs.berkeley.edu/deeprlcourse/)（Sergey Levine） | 第 18 周：Q-learning、REINFORCE、方差缩减 | 策略梯度与 Actor-Critic 的作业；LLM 的强化学习一讲接上第 19 周 |
| 做研究的方法 | [Stanford CS197: Computer Science Research](https://cs197.stanford.edu/) | 整门课的研究方法线：基线、消融、多种子、复现、同伴评审 | 讲座 "Writing a Paper & Picking Projects" 与 "Publication & Peer Review" |
| 药物发现（生物应用） | [TeachOpenCADD](https://projects.volkamerlab.org/teachopencadd/) | Python、pandas、scikit-learn、PyTorch、GNN（第 13 周） | 从分子数据获取与相似性搜索的教程开始，再到其中的深度学习 notebook |
| 深度学习 × 生物 | [*Deep Learning for Biology*](https://www.oreilly.com/library/view/deep-learning-for/9781098168025/)（Charles Ravarani & Natasha Latysheva，O'Reilly，2025） | 整门课 | 挑一个和你实验室最接近的章节；书中用的框架若与课程不同，就把它当作"换一个框架重写你已经懂的模型"的练习 |

最重要的一条建议：**把课上的方法用到你自己实验室的问题上**。选一个你真正关心的问题，按 bit flip 选题、先建基线、≥ 3 个种子、做消融、写实验记录——这套流程不会因为模型变了而过时。

### ③ 个人反思

每人写一份个人反思（第 20 周作业 D 部分，至少 300 字），可以围绕下面的问题：

1. 课程开始前你对"AI 研究"的一个看法，现在变了吗？是哪一周、哪个结果改变了它？
2. 你卡得最久的一处是什么？最后是怎么解决的（调试方法、问人、读文档、AI 助手）？
3. 你的 Capstone 里，哪个结果最让你意外？如果重做，你会改变哪一个设计决定？
4. 在你自己实验室的研究里，哪一个问题可以用这门课的方法来做？写出数据、基线和你会做的第一个实验。
5. 接下来三个月你打算学什么（路线图里的哪一门）？每周投入多少时间？
6. 给这门课的一条建议。

### 全课程回顾：20 周、20 个研究能力关键词

| 周 | 主题 | 研究能力关键词 |
| --- | --- | --- |
| 1 | Python I：变量、容器、循环、条件 | **实验记录**：从第一天起写下做了什么、为什么 |
| 2 | Python II：函数、文件、NumPy、绘图 | **看清数据的形状**：图片就是数组 |
| 3 | Python III：Pandas、调试、类与对象 | **调试**：读报错、打印中间量、缩小问题 |
| 4 | 线性回归与梯度下降 | **优化直觉**：学习率太大、太小各是什么样 |
| 5 | 分类、评估、交叉验证、树模型 | **评估与泄漏**：测试集只用一次 |
| 6 | 无监督学习与基线阶梯 | **基线阶梯与最小实验单元**：先打败简单方法 |
| 7 | 神经网络与 micrograd | **阳性对照**：梯度检查 |
| 8 | PyTorch 与训练循环 | **选题**：bit flip——翻转一个默认假设 |
| 9 | 优化、正则化与调试 | **受控实验**：科学 / 冗余 / 固定超参数，≥ 3 种子 |
| 10 | CNN 与归纳偏置 | **消融**：一次只改一个组件 |
| 11 | Attention 与 Transformer | **机制检验**：去掉位置编码会怎样 |
| 12 | 训练小型语言模型 | **设计评审**：找出最可能让项目失败的假设 |
| 13 | GNN 与预训练概念 | **读论文**：三遍阅读法 |
| 14 | 预训练模型微调与 LoRA | **公平比较**：可训练参数量与调参预算 |
| 15 | 自编码器与 VAE | **复现与差异分析**：差异要给出可检验的原因 |
| 16 | Flow Matching | **可运行的流水线**：先有能跑通的基线 |
| 17 | DDPM 与条件生成 | **受控对比**：两种方法只差一处 |
| 18 | 强化学习 | **方差**：RL 需要更多种子 |
| 19 | AI Agents | **评测集与失败分析**：调通 API 不等于研究 |
| 20 | Capstone 汇报与课程总结 | **汇报与同行评审**：让别人检验你的证据 |

谢谢你走完这 20 周。从今天起，你已经是一名 AI 研究的初学者了——剩下的，是一个接一个地把问题做出来。